# 10 · Other precipitates and the electrolyte

ZnO and Zn(OH)₂ can precipitate with the same rate law as ZHS (`ZnO_on`, `ZnOH2_on`; log K from Herrmann & Horstmann 2024). In ZnSO₄ they saturate at a higher pH than ZHS (about 5.9 and 6.5 against 5.4), so with ZHS present they rarely form.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from znmno2_model.params import Params
from znmno2_model.simulate import run
from znmno2_model.model import Model

def compare(runs, title, cols=("V", "pH_cath")):
    """Plot several runs (dict label -> Result) against capacity."""
    fig, ax = plt.subplots(1, len(cols), figsize=(6 * len(cols), 4.2))
    ax = np.atleast_1d(ax)
    for label, r in runs.items():
        for a, c in zip(ax, cols):
            a.plot(r.column("mAhg"), r.column(c), label=label)
    for a, c in zip(ax, cols):
        a.set(xlabel="capacity [mAh/g]  (decreasing = charge)", ylabel=c)
    ax[0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    return fig


def summary(runs):
    for label, r in runs.items():
        q = r.column("mAhg")
        print(f"{label:28s} exit {r.exit_reason:16s} discharged {q.max():6.1f} mAh/g, final {q[-1]:6.1f} mAh/g")

In [ ]:
runs = {
    "ZHS only (default)": run(Params(steps="cc I=100 Vmin=1.0")),
    "ZHS + ZnO + Zn(OH)2": run(Params(ZnO_on=True, ZnOH2_on=True, steps="cc I=100 Vmin=1.0")),
    "no ZHS, ZnO + Zn(OH)2": run(Params(zhs="off", ZnO_on=True, ZnOH2_on=True, steps="cc I=100 Vmin=1.0")),
    "no precipitate": run(Params(zhs="off", steps="cc I=100 Vmin=1.0")),
}
summary(runs)
compare(runs, "precipitates", cols=("V", "pH_cath"));

## Electrolyte composition

Bernard et al. (2025, electrolyte engineering) varied ZnSO₄ (0.5–3 M) and added H₂SO₄ (0–0.5 M); acid adds a high-voltage plateau at the start of discharge.

In [ ]:
runs = {f"{c} M ZnSO4": run(Params(c_ZnSO4=c, steps="cc I=100 Vmin=1.0")) for c in (0.5, 1.0, 2.0, 3.0)}
summary(runs)
compare(runs, "ZnSO4 concentration");

In [ ]:
runs = {f"{a} M H2SO4": run(Params(c_H2SO4=a, steps="cc I=100 Vmin=1.0")) for a in (0.0, 0.05, 0.1, 0.2)}
summary(runs)
compare(runs, "added H2SO4");

In [ ]:
runs = {f"{m} M MnSO4": run(Params(c_MnSO4=m, steps="cc I=100 Vmin=1.0; cc I=-100 Vmax=1.85", end_on_cutoff=False)) for m in (0.0, 0.05, 0.2, 0.5)}
summary(runs)
compare(runs, "added MnSO4", cols=("V", "Mn_cath_M"));